# Uma rede neural em 20 640 bairros da Califórnia

**Capítulo III.2** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/iii-2-redes-neurais.html).

Você vai:

1. **baixar o conjunto do Kaggle** e conferir o que veio;
2. **derivar** os 8 atributos a partir das 10 colunas do censo, e conferir a sua derivação;
3. medir as **duas linhas de base** — que são o checksum do protocolo, não concorrentes;
4. treinar um **MLP**, e cair na armadilha da escala de propósito para vê-la;
5. **anotar o seu resultado** para levar à aula.

> Roda no Colab sem instalar nada além do `kagglehub`. Se o download falhar — rede fora, cota, conjunto mudou de versão —, o notebook segue com a cópia congelada do repositório e diz que fez isso.

In [ ]:
# Acha o repositório subindo de pasta; no Colab (que não tem), baixa o que precisa do GitHub.
import os, sys, pathlib, urllib.request

RAIZ = None
p = pathlib.Path.cwd()
for _ in range(6):
    if (p / 'ml-zero' / 'etapa-19' / 'dados_kaggle.py').exists():
        RAIZ = p; break
    p = p.parent

BASE = 'https://raw.githubusercontent.com/GHDaru/machinelearning/main/'
ARQUIVOS = ['ml-zero/etapa-19/dados_kaggle.py',
            'ml-zero/etapa-19/mlp.py',
            'ml-zero/etapa-19/rede.py',
            'ml-zero/dados/california/housing_bruto.csv',
            'ml-zero/dados/california/california.csv',
            'ml-zero/dados/california/split.csv']

if RAIZ is None:
    RAIZ = pathlib.Path('conteudo')
    for rel in ARQUIVOS:
        alvo = RAIZ / rel
        alvo.parent.mkdir(parents=True, exist_ok=True)
        if not alvo.exists():
            urllib.request.urlretrieve(BASE + rel, alvo)
    print('Colab: arquivos baixados do GitHub.')

sys.path.insert(0, str(RAIZ / 'ml-zero' / 'etapa-19'))
print('raiz:', RAIZ.resolve())

## 1. Baixar do Kaggle, e conferir o que veio

Este é o trecho que aparece na página do conjunto, sem mudar nada:

```python
import kagglehub
path = kagglehub.dataset_download('camnugent/california-housing-prices')
```

Ele **não pede conta**: o conjunto é público e o `kagglehub` baixa anônimo quando não acha credencial.

**E o passo seguinte é o que quase ninguém dá: conferir.** Download é dependência de rede, de cota e da vontade de terceiro — conjunto no Kaggle ganha revisão nova sem avisar. O repositório guarda uma cópia congelada com `sha256`, e a célula abaixo compara as duas.

In [ ]:
!pip install -q kagglehub

from dados_kaggle import carregar_bruto, baixar_do_kaggle, comparar_com_o_congelado, derivar, conferir

baixado = baixar_do_kaggle()
if baixado:
    for chave, valor in comparar_com_o_congelado(baixado).items():
        print(f'  {chave}: {valor}')

bruto, origem = carregar_bruto(caminho=baixado)
print('\norigem:', origem)
bruto.head()

## 2. O que o arquivo “limpo” não mostra

São **10** colunas, e não 9. Uma delas é texto, e outra tem buracos.

In [ ]:
print('forma:', bruto.shape)
print('\nvalores ausentes:')
print(bruto.isna().sum()[lambda s: s > 0])
print('\ncoluna categórica `ocean_proximity`:')
print(bruto['ocean_proximity'].value_counts())

### Derive você os 8 atributos

Três dos oito são **razões**, e é aí que está o raciocínio. `total_rooms` sozinho diz o tamanho do *setor*, não o tamanho das *casas*: um setor de 5 000 domicílios tem mais cômodos que um de 200 sem que as casas sejam maiores. Dividir por `households` é o que torna a coluna comparável.

```
AveRooms  = total_rooms    / households
AveBedrms = total_bedrooms / households
AveOccup  = population     / households
MedHouseVal = median_house_value / 100000
```

`conferir()` compara o seu resultado com o arquivo congelado, coluna a coluna. É o mesmo papel das linhas de base: **saber que errou antes de tirar conclusão sobre modelo**.

In [ ]:
derivado = derivar(bruto)
resultado = conferir(derivado)
print(resultado)
derivado.describe().T[['mean', 'std', 'min', 'max']].round(3)

> **Olhe a coluna `std` acima antes de seguir.** `Population` tem desvio na casa do milhar e `AveBedrms` na casa do décimo. Guarde isso: é a armadilha do passo 4.

### O achado que só aparece comparando as duas versões

As **207** linhas sem `total_bedrooms` no arquivo do Kaggle **têm valor** no arquivo que o `scikit-learn` distribui. E o valor é inteiro — 217, 279, 1394 — o que descarta imputação por média ou mediana, que daria número quebrado e repetido.

Não é um preenchimento: é dado que **uma das duas cópias perdeu pelo caminho**. Duas versões do “mesmo” conjunto discordam, e nenhuma das duas avisa.

É por isso que a ficha do dado, com origem e `sha256`, não é burocracia.

In [ ]:
import pandas as pd

falta = bruto['total_bedrooms'].isna().to_numpy()
ref = pd.read_csv(RAIZ / 'ml-zero/dados/california/california.csv')
reconstruido = ref.loc[falta, 'AveBedrms'].to_numpy() * bruto.loc[falta, 'households'].to_numpy()

print('linhas sem total_bedrooms no Kaggle:', int(falta.sum()))
print('os mesmos valores no arquivo do scikit-learn:', [int(v) for v in reconstruido.round()[:8]])
print('todos inteiros?', bool((abs(reconstruido - reconstruido.round()) < 0.02).all()))

## 3. As duas linhas de base — um instrumento, não concorrentes

Antes de qualquer rede: prever sempre a **mediana**, e uma **regressão linear**. Ninguém espera que ganhem, e não é para isso que elas existem.

Elas são o **checksum do protocolo**. Se o seu número da regressão linear não for `0.5271`, você não achou um modelo melhor — leu outro arquivo, usou outro recorte ou trocou a métrica. Descobrir isso agora elimina uma classe inteira de discussão que não é sobre modelo nenhum.

O recorte treino/validação/teste vem **gravado em arquivo**, e não sorteado na hora: o embaralhamento muda entre versões da biblioteca, e aí duas turmas de semestres diferentes deixam de ser comparáveis.

In [ ]:
from mlp import carregar, linhas_de_base, treinar, cinco_sementes

dados = carregar()
print({nome: len(y) for nome, (X, y) in dados.items()})

base = linhas_de_base(dados)
for nome, valor in base.items():
    print(f'  {nome:10} MAE {valor:.4f}')

assert abs(base['linear'] - 0.5271) < 1e-3, 'checksum do protocolo falhou — confira o arquivo e o recorte'
print('\nchecksum OK: você está no mesmo protocolo que o resto da turma.')

## 4. O MLP, e a armadilha que não avisa

**Escreva a sua previsão antes de rodar a próxima célula.** O erro sem padronizar vai ficar acima ou abaixo de `0.3878`?

In [ ]:
padronizado = cinco_sementes(dados, padronizar=True)
cru = cinco_sementes(dados, padronizar=False)

print(f"{'':14}{'MAE':>8}{'amplitude':>12}   épocas")
for nome, r in [('padronizado', padronizado), ('cru', cru)]:
    print(f"{nome:14}{r['mediana']:8.4f}{r['amplitude']:12.4f}   {r['epocas']}")
print(f"\nparâmetros: {padronizado['parametros']}  (8×64 + 64 + 64×1 + 1)")

### Leia a terceira coluna antes da primeira

A rede crua **não treinou mais rápido: ela desistiu antes.** O critério de parada viu a perda deixar de melhorar num terreno em que o mesmo passo é grande demais numa direção e minúsculo noutra.

A causa está na tabela do passo 2: `Population` tem desvio padrão 1 132 e `AveBedrms` tem 0,474 — uma razão de **2 390 vezes**.

E o desfecho é o pior possível justamente porque é coerente: o erro cru fica praticamente **empatado com a regressão linear**, o que sustenta a conclusão falsa *“testei, a rede não ganha neste problema”*. Nenhuma exceção foi lançada em momento nenhum.

> **Ausência de exceção não é evidência de correção.**

## 5. A sua vez

Escolha uma configuração diferente da do capítulo. Antes de rodar, **escreva a previsão**: acima ou abaixo de `0.3878`?

Depois compare a diferença **contra a amplitude**. Com 32 unidades a mediana é 0,3852 contra 0,3878 das 64 — diferença de 0,0026, **menor que a amplitude de qualquer uma das duas**. A leitura honesta aí é *“não distingui as duas”*, e não *“32 é melhor”*.

In [ ]:
MINHA_PREVISAO = 0.0      # <-- ESCREVA AQUI antes de rodar
OCULTAS = (32,)           # <-- MEXA AQUI: (32,) · (128,) · (64, 32) · (16, 16, 16)

meu = cinco_sementes(dados, ocultas=OCULTAS, padronizar=True)
print(f'configuração:  {OCULTAS}')
print(f'parâmetros:    {meu["parametros"]}')
print(f'MAE (mediana): {meu["mediana"]:.4f}    amplitude: {meu["amplitude"]:.4f}')
print(f'previsão:      {MINHA_PREVISAO}')

diferenca = abs(meu['mediana'] - 0.3878)
veredito = 'RUÍDO — não dá para distinguir' if diferenca < max(meu['amplitude'], 0.0060) else 'diferença maior que a amplitude'
print(f'\ncontra as 64 do capítulo: diferença {diferenca:.4f} -> {veredito}')

## 6. E a mesma rede, escrita à mão

Tudo acima chama uma rede pronta. O objetivo **O3** do capítulo promete outra coisa: implementar a rede densa em NumPy, do passo para frente ao update. Ela está em [`rede.py`](https://github.com/GHDaru/machinelearning/blob/main/ml-zero/etapa-19/rede.py), ao lado.

A célula abaixo refaz o passo que o capítulo calcula à mão — mesmos nove pesos, mesmo caso — e depois **confere o gradiente contra a diferença finita**. É o teste que separa *“não deu erro”* de *“está correto”*: um sinal trocado não lança exceção nenhuma, a rede treina, a perda desce um pouco, e a culpa cai na taxa de aprendizado.

In [ ]:
from rede import Rede, reproduzir_o_capitulo, conferir_gradiente, XOR_X, XOR_Y, california

c = reproduzir_o_capitulo()
print('o passo do capítulo, refeito aqui:')
print(f"  h1={c['antes']['h'][0]:.4f}  h2={c['antes']['h'][1]:.4f}"
      f"  y={c['antes']['y']:.4f}  E={c['antes']['perda']:.4f}")
print(f"  depois do passo: y={c['depois']['y']:.4f}  E={c['depois']['perda']:.4f}")

r = Rede([2, 3, 1], 'sigmoide', semente=1)
print(f'\nerro do gradiente contra a diferença finita: {conferir_gradiente(r, XOR_X, XOR_Y):.2e}')

print(f"\na rede escrita à mão, nos mesmos bairros: MAE {california()['mae']:.4f}")
print('a biblioteca, no mesmo recorte: 0.3878   ·   a regressão linear: 0.5271')

---

**Leve para a aula:** as duas linhas de base que você obteve, a sua configuração, o número de parâmetros, a mediana e a **amplitude** — e se a sua diferença sobreviveu à amplitude.

O capítulo inteiro está em [machinelearning.ghdaru.com.br/iii-2-redes-neurais.html](https://machinelearning.ghdaru.com.br/iii-2-redes-neurais.html), com o laboratório dos nove pesos e os exercícios corrigidos.